# Constrained CAE optimization with Optuna (RF waveguide design)

Two validation-class optimization workflows that pair the **analytic waveguide S-parameter** helpers (`radia_mcp.radia_ngsolve.waveguide`) with the dependency-free constrained-optimization layer (`radia_mcp.topology_optimization.global_optimizers`) driven by **Optuna** (TPE sampler). Each study **enqueues an analytic initial guess**, runs a reproducible TPE search, and selects the best *feasible* trial (tracking constraint violation). Both run live below.

*Corpus + committed drift-reference JSON kept at `examples/optimization/`; this is the rendered showcase.*

## 1. Quarter-wave dielectric slab matching (constrained TPE)

Minimise reflection `|S11|` of a dielectric slab by choosing `eps_r` and length, with an analytic quarter-wave initial guess **enqueued** into the Optuna TPE study and a feasibility-aware `best_feasible_record` (delivered power / constraint violation).

In [1]:
import sys, os
sys.argv = ["notebook"]

"""Validation-class Optuna study for a waveguide dielectric slab.

The objective is a small CAE-style RF design task: choose a dielectric constant
and slab length in a WR-90-like TE10 guide so that the one-section slab is
nearly transparent at 10 GHz.  Analytic half-wave candidates are enqueued, then
Optuna records a reproducible TPE study.  The final feasible trial is selected
with radia-mcp's dependency-free record helpers.

Optuna is intentionally optional and external to radia-mcp.

Run:

    python examples/optimization/validation_optuna_waveguide_slab.py
"""

from __future__ import annotations

import json
import math
import sys
from pathlib import Path

import optuna


HERE = Path(os.path.join(os.getcwd(), 'notebook.py')).resolve().parent
REPO = HERE.parents[1]
SRC = REPO / "packages" / "radia-mcp" / "src"
# radia_mcp is pip-installed (editable); no sys.path shim needed

from radia_mcp.radia_ngsolve.waveguide import waveguide_dielectric_slab_sparams  # noqa: E402
from radia_mcp.topology_optimization.global_optimizers import (  # noqa: E402
    best_feasible_record,
    constraint_violation,
)


OUT_JSON = HERE / "validation_optuna_waveguide_slab_summary.json"
FREQUENCY = 10.0e9
WIDTH_A = 0.02286
EPS_BOUNDS = (1.5, 4.0)
LENGTH_MM_BOUNDS = (2.0, 30.0)
DELIVERED_TARGET = 0.999
N_TRIALS = 40
SEED = 20260624


def half_wave_length(eps_r: float) -> float:
    slab = waveguide_dielectric_slab_sparams(FREQUENCY, WIDTH_A, eps_r, 0.0)
    return math.pi / slab["beta_slab"]


def evaluate(eps_r: float, length_m: float) -> dict:
    slab = waveguide_dielectric_slab_sparams(FREQUENCY, WIDTH_A, eps_r, length_m)
    delivered = slab["S21_mag"] ** 2
    constraints = [DELIVERED_TARGET - delivered]
    return {
        "objective": slab["S11_mag"],
        "S11_mag": slab["S11_mag"],
        "S21_mag": slab["S21_mag"],
        "delivered_power_fraction": delivered,
        "unitarity": slab["unitarity"],
        "constraints": constraints,
        "constraint_violation": constraint_violation(constraints),
    }


def main() -> int:
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    sampler = optuna.samplers.TPESampler(seed=SEED)
    study = optuna.create_study(direction="minimize", sampler=sampler)

    enqueued = []
    for eps_r in (1.5, 2.2, 3.0, 4.0):
        length_mm = 1.0e3 * half_wave_length(eps_r)
        if LENGTH_MM_BOUNDS[0] <= length_mm <= LENGTH_MM_BOUNDS[1]:
            params = {"eps_r": eps_r, "length_mm": length_mm}
            study.enqueue_trial(params)
            enqueued.append(params)

    def objective(trial):
        eps_r = trial.suggest_float("eps_r", *EPS_BOUNDS)
        length_mm = trial.suggest_float("length_mm", *LENGTH_MM_BOUNDS)
        result = evaluate(eps_r, length_mm * 1.0e-3)
        trial.set_user_attr("S21_mag", result["S21_mag"])
        trial.set_user_attr("delivered_power_fraction", result["delivered_power_fraction"])
        trial.set_user_attr("constraint_delivered", result["constraints"][0])
        trial.set_user_attr("constraint_violation", result["constraint_violation"])
        return result["objective"]

    study.optimize(objective, n_trials=N_TRIALS)

    records = []
    for trial in study.trials:
        constraints = [trial.user_attrs["constraint_delivered"]]
        records.append({
            "number": trial.number,
            "value": trial.value,
            "params": dict(trial.params),
            "constraints": constraints,
            "constraint_violation": constraint_violation(constraints),
            "delivered_power_fraction": trial.user_attrs["delivered_power_fraction"],
            "S21_mag": trial.user_attrs["S21_mag"],
        })
    best_feasible = best_feasible_record(records)
    best_trial = study.best_trial

    summary = {
        "kind": "optuna_waveguide_slab_validation",
        "validation_class": True,
        "optuna_version": optuna.__version__,
        "seed": SEED,
        "n_trials": N_TRIALS,
        "frequency": FREQUENCY,
        "width_a": WIDTH_A,
        "eps_bounds": EPS_BOUNDS,
        "length_mm_bounds": LENGTH_MM_BOUNDS,
        "delivered_target": DELIVERED_TARGET,
        "enqueued": enqueued,
        "study_best": {
            "number": best_trial.number,
            "value": best_trial.value,
            "params": dict(best_trial.params),
        },
        "best_feasible": best_feasible,
        "records": records,
    }
    OUT_JSON.write_text(json.dumps(summary, indent=2), encoding="utf-8")

    print(f"[optuna] version={optuna.__version__}, trials={len(study.trials)}, seed={SEED}")
    print(f"[enqueue] {len(enqueued)} analytic half-wave candidates")
    print(
        f"[best] trial={best_trial.number} |S11|={best_trial.value:.6e} "
        f"eps={best_trial.params['eps_r']:.6g} "
        f"length={best_trial.params['length_mm']:.9f} mm"
    )
    print(
        f"[best feasible] trial={best_feasible['number']} "
        f"|S11|={best_feasible['value']:.6e} "
        f"delivered={best_feasible['delivered_power_fraction']:.12f} "
        f"violation={best_feasible['constraint_violation']:.3e}"
    )
    print(f"[OK] wrote {OUT_JSON}")
    return 0


if True:
    main()


[optuna] version=4.9.0, trials=40, seed=20260624
[enqueue] 4 analytic half-wave candidates
[best] trial=0 |S11|=3.920114e-17 eps=1.5 length=14.490750650 mm
[best feasible] trial=0 |S11|=3.920114e-17 delivered=1.000000000000 violation=0.000e+00
[OK] wrote \\192.168.11.100\work\00_CAE\Radia\01_GitHub\docs\optimization\validation_optuna_waveguide_slab_summary.json


## 2. Bragg-stack filter design (constrained TPE)

Multilayer Bragg reflector: optimise layer parameters to a target stop-band, again seeding the study with analytic half-/quarter-wave guesses and selecting the best feasible trial.

In [2]:
import sys, os
sys.argv = ["notebook"]

"""Validation-class Optuna study for a waveguide Bragg stopband filter.

The objective is a compact CAE-style RF optimization task: tune a 3-period
high/low-permittivity TE10 waveguide stack to suppress transmission at 10 GHz.
Analytic quarter-wave candidates are enqueued, then Optuna records a
reproducible TPE study.  The final feasible trial is selected with radia-mcp's
dependency-free record helpers.

Optuna is intentionally optional and external to radia-mcp.

Run:

    python examples/optimization/validation_optuna_waveguide_bragg_filter.py
"""

from __future__ import annotations

import json
import math
import sys
from pathlib import Path

import optuna


HERE = Path(os.path.join(os.getcwd(), 'notebook.py')).resolve().parent
REPO = HERE.parents[1]
SRC = REPO / "packages" / "radia-mcp" / "src"
# radia_mcp is pip-installed (editable); no sys.path shim needed

from radia_mcp.radia_ngsolve.waveguide import C0, waveguide_cascade_sparams  # noqa: E402
from radia_mcp.topology_optimization.global_optimizers import (  # noqa: E402
    best_feasible_record,
    constraint_violation,
)


OUT_JSON = HERE / "validation_optuna_waveguide_bragg_filter_summary.json"
FREQUENCY = 10.0e9
WIDTH_A = 0.02286
N_PERIODS = 3
REFLECTED_TARGET = 0.95
N_TRIALS = 50
SEED = 20260624

BOUNDS = {
    "eps_hi": (3.0, 6.0),
    "eps_lo": (1.05, 1.8),
    "scale_hi": (0.85, 1.15),
    "scale_lo": (0.85, 1.15),
}

ENQUEUED = [
    {"eps_hi": 4.0, "eps_lo": 1.2, "scale_hi": 1.0, "scale_lo": 1.0},
    {"eps_hi": 6.0, "eps_lo": 1.05, "scale_hi": 1.0, "scale_lo": 1.0},
    {"eps_hi": 5.0, "eps_lo": 1.1, "scale_hi": 1.0, "scale_lo": 1.0},
]


def quarter_wave_length(eps_r: float) -> float:
    k0 = 2.0 * math.pi * FREQUENCY / C0
    kc = math.pi / WIDTH_A
    return math.pi / (2.0 * math.sqrt(eps_r * k0 * k0 - kc * kc))


def sections_from_params(params: dict) -> list[tuple[float, float]]:
    hi = quarter_wave_length(params["eps_hi"]) * params["scale_hi"]
    lo = quarter_wave_length(params["eps_lo"]) * params["scale_lo"]
    sections = []
    for _ in range(N_PERIODS):
        sections.append((hi, params["eps_hi"]))
        sections.append((lo, params["eps_lo"]))
    return sections


def evaluate(params: dict) -> dict:
    sections = sections_from_params(params)
    s = waveguide_cascade_sparams(FREQUENCY, WIDTH_A, sections)
    reflected = s["S11_mag"] ** 2
    constraints = [REFLECTED_TARGET - reflected]
    return {
        "objective": s["S21_mag"],
        "S11_mag": s["S11_mag"],
        "S21_mag": s["S21_mag"],
        "reflected_power": reflected,
        "transmitted_power": s["S21_mag"] ** 2,
        "unitarity": s["unitarity"],
        "max_section_length_mm": 1.0e3 * max(length for length, _ in sections),
        "sections_mm": [1.0e3 * length for length, _ in sections],
        "constraints": constraints,
        "constraint_violation": constraint_violation(constraints),
    }


def _trial_params(trial) -> dict:
    params = {
        "eps_hi": trial.suggest_float("eps_hi", *BOUNDS["eps_hi"]),
        "eps_lo": trial.suggest_float("eps_lo", *BOUNDS["eps_lo"]),
        "scale_hi": trial.suggest_float("scale_hi", *BOUNDS["scale_hi"]),
        "scale_lo": trial.suggest_float("scale_lo", *BOUNDS["scale_lo"]),
    }
    if params["eps_hi"] <= params["eps_lo"]:
        # Outside the intended high/low stack ordering; keep it unattractive.
        params["eps_hi"], params["eps_lo"] = params["eps_lo"], params["eps_hi"]
    return params


def main() -> int:
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    sampler = optuna.samplers.TPESampler(seed=SEED)
    study = optuna.create_study(direction="minimize", sampler=sampler)

    for params in ENQUEUED:
        study.enqueue_trial(params)

    def objective(trial):
        params = _trial_params(trial)
        result = evaluate(params)
        for key in ("S11_mag", "S21_mag", "reflected_power", "transmitted_power",
                    "unitarity", "max_section_length_mm", "constraint_violation"):
            trial.set_user_attr(key, result[key])
        trial.set_user_attr("constraint_reflected", result["constraints"][0])
        trial.set_user_attr("sections_mm", result["sections_mm"])
        return result["objective"]

    study.optimize(objective, n_trials=N_TRIALS)

    records = []
    for trial in study.trials:
        constraints = [trial.user_attrs["constraint_reflected"]]
        records.append({
            "number": trial.number,
            "value": trial.value,
            "params": dict(trial.params),
            "constraints": constraints,
            "constraint_violation": constraint_violation(constraints),
            "S11_mag": trial.user_attrs["S11_mag"],
            "S21_mag": trial.user_attrs["S21_mag"],
            "reflected_power": trial.user_attrs["reflected_power"],
            "transmitted_power": trial.user_attrs["transmitted_power"],
            "unitarity": trial.user_attrs["unitarity"],
            "max_section_length_mm": trial.user_attrs["max_section_length_mm"],
            "sections_mm": trial.user_attrs["sections_mm"],
        })

    best_feasible = best_feasible_record(records)
    best_trial = study.best_trial
    enqueued_records = [records[i] for i in range(len(ENQUEUED))]
    max_unitarity_error = max(abs(r["unitarity"] - 1.0) for r in records)

    summary = {
        "kind": "optuna_waveguide_bragg_filter_validation",
        "validation_class": True,
        "optuna_version": optuna.__version__,
        "seed": SEED,
        "n_trials": N_TRIALS,
        "frequency": FREQUENCY,
        "width_a": WIDTH_A,
        "n_periods": N_PERIODS,
        "bounds": BOUNDS,
        "reflected_target": REFLECTED_TARGET,
        "enqueued": ENQUEUED,
        "enqueued_records": enqueued_records,
        "study_best": {
            "number": best_trial.number,
            "value": best_trial.value,
            "params": dict(best_trial.params),
        },
        "best_feasible": best_feasible,
        "max_unitarity_error": max_unitarity_error,
        "records": records,
    }

    assert enqueued_records[0]["constraint_violation"] == 0.0
    assert best_feasible["constraint_violation"] == 0.0
    assert best_feasible["reflected_power"] >= REFLECTED_TARGET
    assert best_feasible["S21_mag"] < 0.10
    assert best_feasible["value"] <= enqueued_records[0]["value"]
    assert max_unitarity_error < 1.0e-12

    OUT_JSON.write_text(json.dumps(summary, indent=2), encoding="utf-8")

    print(f"[optuna] version={optuna.__version__}, trials={len(study.trials)}, seed={SEED}")
    print(f"[enqueue] {len(ENQUEUED)} analytic quarter-wave candidates")
    for rec in enqueued_records:
        print(
            f"  trial={rec['number']:2d} |S21|={rec['S21_mag']:.6f} "
            f"reflected={rec['reflected_power']:.6f} "
            f"eps_hi={rec['params']['eps_hi']:.4g} eps_lo={rec['params']['eps_lo']:.4g}"
        )
    print(
        f"[best feasible] trial={best_feasible['number']} "
        f"|S21|={best_feasible['S21_mag']:.6f} "
        f"|S11|={best_feasible['S11_mag']:.6f} "
        f"reflected={best_feasible['reflected_power']:.12f} "
        f"violation={best_feasible['constraint_violation']:.3e}"
    )
    print(f"[OK] wrote {OUT_JSON}")
    return 0


if True:
    main()


[optuna] version=4.9.0, trials=50, seed=20260624
[enqueue] 3 analytic quarter-wave candidates
  trial= 0 |S21|=0.198360 reflected=0.960653 eps_hi=4 eps_lo=1.2
  trial= 1 |S21|=0.074178 reflected=0.994498 eps_hi=6 eps_lo=1.05
  trial= 2 |S21|=0.111927 reflected=0.987472 eps_hi=5 eps_lo=1.1
[best feasible] trial=1 |S21|=0.074178 |S11|=0.997245 reflected=0.994497688055 violation=0.000e+00
[OK] wrote \\192.168.11.100\work\00_CAE\Radia\01_GitHub\docs\optimization\validation_optuna_waveguide_bragg_filter_summary.json
